In [1]:
import pandas as pd
import tensorflow as tf
import os

2026-04-28 08:14:11.324499: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777364051.339699      10 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777364051.344519      10 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 08:14:11.364834: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
#Hyperparameters
output_dir = "./"
test_dir = "../input/plant-pathology-2021-fgvc8/test_images/"
model_dir ="../input/dense-e1/epoch-1"

image_dims = (300,300,3)
data_set = pd.read_csv("../input/plant-pathology-2021-fgvc8/train.csv")
df_labels = data_set['labels']
one_hot = df_labels.str.get_dummies(sep=" ")
dataset_labels = one_hot.columns.to_list()

In [3]:
if __name__ == '__main__':

    # Initialization of model and data creator class.
    model = tf.keras.models.load_model(model_dir)
    images_path_list = sorted(list(os.listdir(test_dir)))
    def test_on_sub (index):
        input_img = tf.io.read_file(test_dir+images_path_list[index])
        image = tf.io.decode_image(contents=input_img, channels=3, dtype=tf.dtypes.float32)
        tensor_image = tf.image.resize(image, [image_dims[0], image_dims[1]])
        name_jpg = images_path_list[index].split(os.path.sep)[-1]
        return name_jpg, tf.expand_dims(tensor_image, axis=0)
    
    values = []
    for i in range(len(images_path_list)):
        name, images = test_on_sub(index=i)
        images = images * 255
        test_values = model.call(inputs=images)
        print(test_values)
        index_values = [i for i, v in enumerate(test_values[0]) if v > 0.7]
        classes = dataset_labels
        classes_img = ""
        for i in index_values:
            classes_img = str(classes[i])+" "+classes_img
        values.append([name, classes_img])
    csv_pd = pd.DataFrame(values, columns=['image', 'labels'], index=None)
    csv_pd.to_csv(output_dir + 'submission.csv',index=False)

ValueError: File format not supported: filepath=../input/dense-e1/epoch-1. Keras 3 only supports V3 `.keras` files and legacy H5 format files (`.h5` extension). Note that the legacy SavedModel format is not supported by `load_model()` in Keras 3. In order to reload a TensorFlow SavedModel as an inference-only layer in Keras 3, use `keras.layers.TFSMLayer(../input/dense-e1/epoch-1, call_endpoint='serving_default')` (note that your `call_endpoint` might have a different name).